# 01 - Data check and extra shots (penalties + free kicks)

**Goal:** find the shot types that are missing from `shots_clean.csv` and add them
to the data without changing `clean.py`.

**How to use this notebook:** run the cells from top to bottom (`Shift+Enter`).
Read the explanation above each cell and try to interpret the output yourself.
Before running, make sure the kernel (top right) is the **`.venv`** environment.

## 1. Setup
We add the `src/` folder to Python's search path so the notebook can import
`clean.py` (the existing cleaning code) and `extra_shots.py` (our addition).

In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook lives in 'notebooks/' and the code in 'src/'. Add ../src to the path.
sys.path.append(str(Path("..").resolve() / "src"))

from clean import RAW_PATH, load_events, clean_shots           # existing code, unchanged
from extra_shots import extract_extra_shots, build_all_shots   # our addition

pd.set_option("display.max_columns", 30)

## 2. Load the raw events and the existing shots
- `events`: 643,150 raw events (passes, duels, shots, fouls, ...)
- `shots`: the shot table produced by `clean_shots` from `clean.py`

In [2]:
events = load_events(RAW_PATH)
shots = clean_shots(events)

print("Raw events:", len(events))
print("Shots (clean.py):", len(shots))
shots.head()

Raw events: 643150
Shots (clean.py): 8451


,playerId,matchId,teamId,matchPeriod,eventSec,id,start_x,start_y,is_goal,left_foot,right_foot,header,counter_attack
0,25413,2499719,1609,1H,94.595788,177959212,88,41,True,False,True,False,False
1,26150,2499719,1631,1H,179.854785,177959247,85,52,False,True,False,False,False
2,14763,2499719,1631,1H,254.745027,177959280,96,52,True,False,False,True,False
3,7868,2499719,1609,1H,425.824035,177959289,81,33,False,True,False,False,False
4,7868,2499719,1609,1H,815.462015,177959429,75,30,False,False,True,False,False


## 3. The problem: are some shots missing?
`clean.py` keeps only rows with `eventName == "Shot"`.
But in the Wyscout data, penalties and direct free-kick shots are stored under the
**`Free Kick`** event. Let's see this for ourselves:

In [3]:
# Sub-event types of the Free Kick event
events[events["eventName"] == "Free Kick"]["subEventName"].value_counts()

subEventName
Throw in           17050
Free Kick           7279
Goal kick           6061
Corner              3910
Free kick cross     1693
Free kick shot       350
Penalty               80
Name: count, dtype: int64

**Interpretation:** `Penalty` (80) and `Free kick shot` (350) are shots, but
`clean.py` does not pick them up. `Throw in`, `Corner`, `Goal kick`, `Free Kick`
(a short free kick) and `Free kick cross` are not shots, so we leave them out.

Let's also check that every `Shot` event really has the sub-event type `Shot`:

In [4]:
events[events["eventName"] == "Shot"]["subEventName"].value_counts()

subEventName
Shot    8451
Name: count, dtype: int64

## 4. The fix: `extract_extra_shots`
The function in `src/extra_shots.py` does the following:

1. From the `Free Kick` events, it selects only `Penalty` and `Free kick shot`.
2. It stores the type in a `shot_type` column (`penalty` / `free_kick_shot`).
   This happens **first**, because the next step removes `subEventName`.
3. It reuses `drop_unused_columns` and `parse_tags` from `clean.py` **as they are**,
   so the output has exactly the same columns as `shots_clean.csv`.

`build_all_shots` then combines the regular shots (`shot_type = "shot"`) with the
extra shots. We did not edit `clean.py`, we only call its functions.

In [5]:
extra = extract_extra_shots(events)
print(extra.shape)
extra.head()

(430, 14)


,playerId,matchId,teamId,matchPeriod,eventSec,id,start_x,start_y,is_goal,shot_type,left_foot,right_foot,header,counter_attack
0,38021,2499720,1625,1H,995.894670,178147638,76,33,False,free_kick_shot,False,True,False,False
1,38021,2499720,1625,1H,1873.311156,178147796,77,29,False,free_kick_shot,False,True,False,False
2,8032,2499721,1610,2H,951.436722,178123230,72,64,False,free_kick_shot,True,False,False,False
3,9623,2499721,1646,2H,2664.038744,178123617,83,49,False,free_kick_shot,True,False,False,False
4,454,2499722,1628,2H,1715.579428,178205147,70,54,False,free_kick_shot,True,False,False,False


In [6]:
all_shots = build_all_shots(events)
all_shots["shot_type"].value_counts()

shot_type
shot              8451
free_kick_shot     350
penalty             80
Name: count, dtype: int64

## 5. Validation (does our new code work correctly?)
We check three things:
1. Are the columns the same as in `shots_clean` (plus `shot_type`)?
2. Are the regular shot rows unchanged?
3. Do the numbers make sense? Penalties should be close to goal and central, with a high goal rate.

In [7]:
# 1) Columns
print(list(all_shots.columns))
print("Same columns:", list(all_shots.columns) == list(shots.columns) + ["shot_type"])

# 2) Regular shots are untouched
normal = all_shots[all_shots["shot_type"] == "shot"].drop(columns="shot_type").reset_index(drop=True)
print("Regular shots unchanged:", normal.equals(shots))

['playerId', 'matchId', 'teamId', 'matchPeriod', 'eventSec', 'id', 'start_x', 'start_y', 'is_goal', 'left_foot', 'right_foot', 'header', 'counter_attack', 'shot_type']
Same columns: True
Regular shots unchanged: True


In [8]:
# 3) Summary by type: count, goals, goal rate, median location
all_shots.groupby("shot_type").agg(
    shots=("is_goal", "size"),
    goals=("is_goal", "sum"),
    goal_rate=("is_goal", "mean"),
    median_x=("start_x", "median"),
    median_y=("start_y", "median"),
).round(3)

,shots,goals,goal_rate,median_x,median_y
shot_type,,,,,
free_kick_shot,350,18,0.051,76.0,47.5
penalty,80,56,0.700,90.0,50.0
shot,8451,914,0.108,87.0,49.0


**Interpretation:**
- **Penalty:** 80 shots, goal rate ~70%, far above regular shots (10.8%). The median
  location is x=90, y=50, which is the penalty spot, as expected.
- **Free-kick shot:** 350 shots, goal rate ~5%, *lower* than regular shots. They are
  taken from further away (median x=76).
- The three types behave **very differently**. When building the model (Day 4) we
  will probably need to handle penalties separately, which is why we kept `shot_type`.

## 6. Save
We save the combined table as `data/processed/shots_all.csv`.
`shots_clean.csv` (the existing file) stays as it is.

In [9]:
OUT_PATH = Path("..") / "data" / "processed" / "shots_all.csv"
all_shots.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH.resolve(), "|", all_shots.shape)

Saved: /Users/alicanaksel/Desktop/xG-Analyzer/xG-Analyzer/data/processed/shots_all.csv | (8881, 14)


## 7. Next step
Shot distribution and goal rate plots (`02_eda_plots.ipynb`). Before starting, decide
as a team whether the plots show penalties and free kicks separately or together.
The `shot_type` column allows both.